In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os
import copy
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from PIL import Image
from torchvision import models, transforms

In [6]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


path set up 

In [8]:
AUTHOR_ROOT = "/content/drive/MyDrive/scenario23_author_files"   # change if folder name differs
DATA_ROOT   = "/content/dataset/scenario23_dev"                  # same raw dataset path as before

train_csv_author = os.path.join(AUTHOR_ROOT, "scenario23_img_beam_train.csv")
val_csv_author   = os.path.join(AUTHOR_ROOT, "scenario23_img_beam_val.csv")
test_csv_author  = os.path.join(AUTHOR_ROOT, "scenario23_img_beam_test.csv")

train_csv_fixed = "/content/scenario23_img_beam_train_fixed.csv"
val_csv_fixed   = "/content/scenario23_img_beam_val_fixed.csv"
test_csv_fixed  = "/content/scenario23_img_beam_test_fixed.csv"

print(os.path.exists(train_csv_author), train_csv_author)
print(os.path.exists(val_csv_author), val_csv_author)
print(os.path.exists(test_csv_author), test_csv_author)
print("DATA_ROOT exists:", os.path.exists(DATA_ROOT))

False /content/drive/MyDrive/scenario23_author_files/scenario23_img_beam_train.csv
False /content/drive/MyDrive/scenario23_author_files/scenario23_img_beam_val.csv
False /content/drive/MyDrive/scenario23_author_files/scenario23_img_beam_test.csv
DATA_ROOT exists: False


In [9]:
import os

drive_root = "/content/drive/MyDrive"

matches = []
for root, dirs, files in os.walk(drive_root):
    if "scenario23_img_beam_train.csv" in files:
        matches.append(root)

print("Candidate folders:")
for m in matches:
    print(m)

Candidate folders:
/content/drive/MyDrive/Image beam


In [10]:
AUTHOR_ROOT = "/content/drive/MyDrive/Image beam"

In [11]:
train_csv_author = os.path.join(AUTHOR_ROOT, "scenario23_img_beam_train.csv")
val_csv_author   = os.path.join(AUTHOR_ROOT, "scenario23_img_beam_val.csv")
test_csv_author  = os.path.join(AUTHOR_ROOT, "scenario23_img_beam_test.csv")

print(os.path.exists(train_csv_author), train_csv_author)
print(os.path.exists(val_csv_author), val_csv_author)
print(os.path.exists(test_csv_author), test_csv_author)

True /content/drive/MyDrive/Image beam/scenario23_img_beam_train.csv
True /content/drive/MyDrive/Image beam/scenario23_img_beam_val.csv
True /content/drive/MyDrive/Image beam/scenario23_img_beam_test.csv


In [13]:
zip_path = "/content/drive/MyDrive/scenario23_dev_w_resources.zip"
print("Exists:", os.path.exists(zip_path))

Exists: True


In [15]:
import zipfile

extract_dir = "/content/dataset"

os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_dir)

print("Unzipped to:", extract_dir)

Unzipped to: /content/dataset


In [17]:
for root, dirs, files in os.walk(extract_dir):
    if "scenario23" in root.lower():
        print(root)

/content/dataset/scenario23_dev
/content/dataset/scenario23_dev/unit2
/content/dataset/scenario23_dev/unit2/altitude
/content/dataset/scenario23_dev/unit2/distance
/content/dataset/scenario23_dev/unit2/height
/content/dataset/scenario23_dev/unit2/y_speed
/content/dataset/scenario23_dev/unit2/pitch
/content/dataset/scenario23_dev/unit2/speed
/content/dataset/scenario23_dev/unit2/roll
/content/dataset/scenario23_dev/unit2/z_speed
/content/dataset/scenario23_dev/unit2/x_speed
/content/dataset/scenario23_dev/unit2/GPS_data
/content/dataset/scenario23_dev/unit1
/content/dataset/scenario23_dev/unit1/mmWave_data
/content/dataset/scenario23_dev/unit1/camera_data
/content/dataset/scenario23_dev/unit1/GPS_data
/content/dataset/scenario23_dev/resources
/content/dataset/scenario23_dev/resources/bbox_labels_final


check zip in a better approach 

In [19]:
drive_root = "/content/drive/MyDrive"

zip_matches = []
for root, dirs, files in os.walk(drive_root):
    for f in files:
        if f.lower().endswith(".zip") and "scenario23" in f.lower():
            zip_matches.append(os.path.join(root, f))

print("Candidate ZIP files:")
for z in zip_matches:
    print(z)

Candidate ZIP files:
/content/drive/MyDrive/scenario23_dev_w_resources.zip


In [20]:
ZIP_PATH = "/content/drive/MyDrive/scenario23_dev_w_resources.zip"   # replace this
print("ZIP exists:", os.path.exists(ZIP_PATH), ZIP_PATH)

ZIP exists: True /content/drive/MyDrive/scenario23_dev_w_resources.zip


In [21]:
EXTRACT_ROOT = "/content/dataset"
os.makedirs(EXTRACT_ROOT, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(EXTRACT_ROOT)

print("Unzip done")

Unzip done


detect real dataroot after unzip 

In [22]:
candidate_roots = []
for root, dirs, files in os.walk(EXTRACT_ROOT):
    if "scenario23.csv" in files:
        candidate_roots.append(root)

print("Candidate DATA_ROOT folders:")
for c in candidate_roots:
    print(c)

Candidate DATA_ROOT folders:
/content/dataset/scenario23_dev


In [23]:
train_csv_fixed = "/content/scenario23_img_beam_train_fixed.csv"
val_csv_fixed   = "/content/scenario23_img_beam_val_fixed.csv"
test_csv_fixed  = "/content/scenario23_img_beam_test_fixed.csv"

if os.path.exists(train_csv_fixed) and os.path.exists(val_csv_fixed) and os.path.exists(test_csv_fixed):
    print("Using existing fixed CSV files")
else:
    print("Building fixed CSV files from author CSV + extracted dataset")

    image_map = {}
    for root, dirs, files in os.walk(DATA_ROOT):
        for f in files:
            if f.lower().endswith((".jpg", ".jpeg", ".png")):
                image_map[f] = os.path.join(root, f)

    print("Indexed images:", len(image_map))

    def fix_img_csv_paths(csv_path, out_path):
        dfc = pd.read_csv(csv_path).copy()
        path_col = dfc.columns[1]
        dfc[path_col] = dfc[path_col].apply(
            lambda x: image_map.get(os.path.basename(str(x).strip()), str(x))
        )
        dfc.to_csv(out_path, index=False)
        print("Saved:", out_path)
        return dfc

    fix_img_csv_paths(train_csv_author, train_csv_fixed)
    fix_img_csv_paths(val_csv_author, val_csv_fixed)
    fix_img_csv_paths(test_csv_author, test_csv_fixed)

Building fixed CSV files from author CSV + extracted dataset
Indexed images: 11387
Saved: /content/scenario23_img_beam_train_fixed.csv
Saved: /content/scenario23_img_beam_val_fixed.csv
Saved: /content/scenario23_img_beam_test_fixed.csv


In [24]:
train_df = pd.read_csv(train_csv_fixed)
val_df   = pd.read_csv(val_csv_fixed)
test_df  = pd.read_csv(test_csv_fixed)

print("Train:", train_df.shape)
print("Val  :", val_df.shape)
print("Test :", test_df.shape)
print(train_df.head())

Train: (6832, 3)
Val  : (3416, 3)
Test : (1139, 3)
   index                                          unit1_rgb  unit1_beam
0   3532  /content/dataset/scenario23_dev/unit1/camera_d...          17
1   2224  /content/dataset/scenario23_dev/unit1/camera_d...          14
2   9416  /content/dataset/scenario23_dev/unit1/camera_d...          17
3   8510  /content/dataset/scenario23_dev/unit1/camera_d...          20
4   6877  /content/dataset/scenario23_dev/unit1/camera_d...          17


quick image path check 

In [25]:
for p in train_df.iloc[:10, 1].tolist():
    print(p, os.path.exists(str(p)))

/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_3532_17_08_22.jpg True
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_2224_17_04_35.jpg True
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_10033_17_56_07.jpg True
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_9127_17_53_50.jpg True
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_7494_17_48_19.jpg True
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_3825_17_09_10.jpg True
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_2258_17_04_41.jpg True
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_2121_17_04_20.jpg True
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_11899_18_02_04.jpg True
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_8210_17_50_08.jpg True


dataset

In [26]:
class ImageBeamDataset(Dataset):
    def __init__(self, csv_path, transform=None):
        self.df = pd.read_csv(csv_path).reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row.iloc[1]
        label = int(row.iloc[2])

        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)

        return img, label

In [27]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    )
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    )
])

In [28]:
batch_size = 32
val_batch_size = 32

train_loader = DataLoader(
    ImageBeamDataset(train_csv_fixed, transform=train_transform),
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    ImageBeamDataset(val_csv_fixed, transform=test_transform),
    batch_size=val_batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    ImageBeamDataset(test_csv_fixed, transform=test_transform),
    batch_size=val_batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

sanity check 

In [29]:
imgs, labels = next(iter(train_loader))
print(imgs.shape, labels.shape)
print(labels[:10])

torch.Size([32, 3, 224, 224]) torch.Size([32])
tensor([17,  4,  2, 26, 15, 20, 15, 30, 11, 17])


In [30]:
def evaluate_topk(model, loader, device, ks=(1, 2, 3, 5)):
    model.eval()
    total = 0
    correct = {k: 0 for k in ks}

    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(imgs)
            max_k = max(ks)
            _, pred = torch.topk(outputs, k=max_k, dim=1)
            pred = pred.t()

            total += labels.size(0)
            for k in ks:
                correct[k] += pred[:k].eq(labels.view(1, -1)).sum().item()

    return {f"top{k}": 100.0 * correct[k] / total for k in ks}

train the epoch 

In [31]:
def train_model(
    model,
    train_loader,
    val_loader,
    device,
    epochs=10,
    lr=1e-4,
    weight_decay=1e-4,
    milestones=(5, 8),
    save_path="/content/drive/MyDrive/best_model.pth"
):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=list(milestones), gamma=0.1)

    best_top1 = -1
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        total_train = 0

        for step, (imgs, labels) in enumerate(train_loader, start=1):
            imgs = imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            bs = labels.size(0)
            running_loss += loss.item() * bs
            total_train += bs

            if step % 50 == 0:
                print(f"Epoch {epoch:02d} Step {step}/{len(train_loader)} Loss {loss.item():.4f}")

        scheduler.step()

        train_loss = running_loss / total_train
        val_metrics = evaluate_topk(model, val_loader, device, ks=(1, 2, 3, 5))

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            **val_metrics
        })

        print(
            f"Epoch {epoch:02d} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Top1: {val_metrics['top1']:.2f} | "
            f"Top2: {val_metrics['top2']:.2f} | "
            f"Top3: {val_metrics['top3']:.2f} | "
            f"Top5: {val_metrics['top5']:.2f}"
        )

        if val_metrics["top1"] > best_top1:
            best_top1 = val_metrics["top1"]
            torch.save(copy.deepcopy(model.state_dict()), save_path)
            print("Saved best model")

    return pd.DataFrame(history)

In [ ]:
vgg_model = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
vgg_model.classifier[6] = nn.Linear(vgg_model.classifier[6].in_features, 64)
vgg_model = vgg_model.to(device)

hist_vgg = train_model(
    model=vgg_model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    epochs=10,
    lr=1e-4,
    weight_decay=1e-4,
    milestones=(5, 8),
    save_path="/content/drive/MyDrive/best_vgg16.pth"
)

Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:08<00:00, 69.0MB/s] 


Epoch 01 Step 50/214 Loss 1.3575
Epoch 01 Step 100/214 Loss 0.7751
Epoch 01 Step 150/214 Loss 0.6502
Epoch 01 Step 200/214 Loss 0.4354
Epoch 01 | Train Loss: 1.1982 | Top1: 81.38 | Top2: 95.08 | Top3: 98.71 | Top5: 99.47
Saved best model
Epoch 02 Step 50/214 Loss 0.4324
Epoch 02 Step 100/214 Loss 0.3780
Epoch 02 Step 150/214 Loss 0.5024
Epoch 02 Step 200/214 Loss 0.2717
Epoch 02 | Train Loss: 0.4794 | Top1: 86.42 | Top2: 97.69 | Top3: 99.12 | Top5: 99.62
Saved best model
Epoch 03 Step 50/214 Loss 0.2419
Epoch 03 Step 100/214 Loss 0.2974
